In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# LINK TO KAGGLE DATA :: https://www.kaggle.com/competitions/playground-series-s5e7/data

import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

from sklearn.impute import IterativeImputer


import warnings
warnings.filterwarnings("ignore")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Read csv data

In [ ]:
#read in the data from the train csv file and print
trainData= pd.read_csv("/kaggle/input/playground-series-s5e7/train.csv")
print(trainData)

In [ ]:
#read in the data from the test csv file and print
testData = pd.read_csv("/kaggle/input/playground-series-s5e7/test.csv")
print(testData)

# DISPLAY SUMMARY INFO ABOUT THE FILES

In [ ]:
# display info about the data, with datatypes
trainData = trainData.info()
print(trainData)

In [ ]:
testData = testData.info()
print(testData)

# TOTAL RECORD SIZE AND STATS METRICS

In [ ]:
# display the total count of records, min, max value, view the std dev and mean
trainData = trainData.describe()
print(trainData)

In [ ]:
testData = testData.describe()
print(trainData)

# FEATURE ENGINEERING

In [ ]:
# map strings to numerical data for the model training Set
trainData['Stage_fear'] = trainData['Stage_fear'].map({ "No" : 1, "Yes" : 0 })
trainData['Drained_after_socializing'] = trainData['Drained_after_socializing'].map({ "No" : 1, "Yes" : 0 })
trainData['Personality'] = trainData['Personality'].map({ "Extrovert" : 1, "Introvert" : 0 })

print("Done numerical mapping")
print(trainData)

In [ ]:
# map strings to numerical data for the model training Set
testData['Stage_fear'] = testData['Stage_fear'].map({ "No" : 1, "Yes" : 0 })
testData['Drained_after_socializing'] = testData['Drained_after_socializing'].map({ "No" : 1, "Yes" : 0 })

print(testData)

# HANDLE MISSING VALUES

In [ ]:
# TRAINING SET
# HANDLE MISSING VALUES BY PREDICTING FROM AVAILABLE SET OR JUST SET DEFAULT VALUE TO 0
imputer = IterativeImputer(max_iter=50, random_state=5)
imputer = imputer.fit_transform(trainData)
trainImputer = pd.DataFrame(imputer, columns=trainData.columns)
print(trainImputer)

In [ ]:
# ensure null is zero using Imputer
trainImputer = trainImputer.isnull().sum()
print(trainImputer)

In [ ]:
# visualise the correlation between 2 attributes from the data set
trainSea = sns.pairplot(x="Time_spent_Alone", y="Post_frequency", trainData, alpha=0.5)
print(trainSea)

In [ ]:
# TESTING SET
# HANDLE MISSING VALUES BY PREDICTING FROM AVAILABLE SET OR JUST SET DEFAULT VALUE TO 0
imputer = IterativeImputer(max_iter=50, random_state=6)
imputer = imputer.fit_transform(testData)
testImputer = pd.DataFrame(imputer, columns=testData.columns)
print(testImputer)

In [ ]:
testImputer = testImputer.isnull().sum()
print(testImputer)

In [ ]:
# visualise the correlation between 2 attributes from the data set
testSea = sns.jointplot(x="Social_event_attendance", y="Friends_circle_size", data=testImputer, alpha=0.5)
print(testSea)

# PREPARE FOR FIT AND PREDICT

In [ ]:
# mask irrelevant attributes
X = trainImputer.drop(columns=["id", "Personality"])
y = trainImputer["Personality"]
print("Dropped layers pass")

In [ ]:
# instance KMeans clustering
kMeans = KMeans(n_clusters=3, random_state=7)
clusterTrue = kMeans.fit_predict(X)

In [ ]:
Xclusters = pd.concat([pd.DataFrame(X), pd.get_dummies(clusterTrue, prefix='cluster')], axis=1)

In [ ]:
# Split data for training and validaton for accuracy
X_train, X_valid, y_train, y_valid = train_test_split(Xclusters, y, test_size=0.4, random_state=4) 
print("Done splitting train set")

In [ ]:
logReg = LogisticRegression(max_iter=1500)
logReg  = logReg.fit(X_train, y_train)
print(logReg)

In [ ]:
# conduct inference check on the validation set
_pred = logReg.predict(X_valid)
accuracy = accuracy_score(_pred, y_valid)
print("Accuracy: ", accuracy)

# Accuracy:  0.97165991902834

In [ ]:
testImputer = testImputer.drop(columns={"id"})
_testPred = logReg.predict(testImputer)

# CREATE SUBMISSION FILE

In [ ]:
reverseMap = {1: "Extrovert" , 0: "Introvert"}
conductMap = [reverseMap[p] for p in _testPred]

submission = pd.DataFrame({
    "id":testData["id"],
    "Personality": conductMap
})
submission.to_csv("submission.csv", index= False)